# Gaussian Process

In [5]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

torch.set_default_dtype(torch.float64)   # float64 for a stable Cholesky (so CPU: MPS has no float64)

X = pd.read_csv("../X_train_log_knn.csv", index_col="id")
X_test = pd.read_csv("../X_test_log_knn.csv", index_col="id")
y = pd.read_csv("../y_train.csv", index_col="id").loc[X.index, "y"]
fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X.index, "fold"]
print(f"train {X.shape} | test {X_test.shape} | y {y.shape}")

train (1152, 216) | test (776, 216) | y (1152,)


In [6]:
# Exact GP with an ARD RBF kernel: k(a, b) = s² · exp(-½ Σ_d (a_d - b_d)² / ℓ_d²), plus noise σ² on the diagonal.
# One length scale ℓ_d per feature; all hyperparameters are fitted by maximizing the marginal likelihood (X and y standardized).

def kernel(A, B, log_ls, log_sf):
    A, B = A / log_ls.exp(), B / log_ls.exp()
    sq = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T   # squared distances, scaled per feature
    return (2 * log_sf).exp() * torch.exp(-0.5 * sq.clamp_min(0))

def fit_gp(X_tr, y_tr, steps=600, lr=0.05):
    scaler = StandardScaler().fit(X_tr)
    Xt = torch.tensor(scaler.transform(X_tr))
    y_mean, y_std = y_tr.mean(), y_tr.std()
    yt = torch.tensor(((y_tr - y_mean) / y_std).to_numpy())[:, None]
    n, d = Xt.shape

    # start from the SVR's kernel width: gamma ≈ 0.005 -> ℓ = 1 / sqrt(2 · gamma) = 10 for every feature
    log_ls = torch.full((d,), np.log(10.0), requires_grad=True)
    log_sf = torch.tensor(0.0, requires_grad=True)            # signal sd s = 1 (y is standardized)
    log_sn = torch.tensor(np.log(0.5), requires_grad=True)    # noise sd σ = 0.5 (the SVR leaves ~30% of the variance)
    opt = torch.optim.Adam([log_ls, log_sf, log_sn], lr=lr)

    def factor():
        K = kernel(Xt, Xt, log_ls, log_sf) + ((2 * log_sn).exp() + 1e-4) * torch.eye(n)
        L = torch.linalg.cholesky(K)
        return L, torch.cholesky_solve(yt, L)

    history = []
    for step in range(steps):
        opt.zero_grad()
        L, alpha = factor()
        nll = 0.5 * (yt * alpha).sum() + L.diagonal().log().sum()   # negative log marginal likelihood (constant dropped)
        nll.backward()
        opt.step()
        history.append(nll.item())
    print("  nll every 100 steps:", [round(v, 1) for v in history[::100] + history[-1:]])

    with torch.no_grad():
        L, alpha = factor()
    return {"scaler": scaler, "X": Xt, "alpha": alpha, "log_ls": log_ls.detach(), "log_sf": log_sf.detach(),
            "y_mean": y_mean, "y_std": y_std}

def predict_gp(gp, X_new):
    Xn = torch.tensor(gp["scaler"].transform(X_new))
    with torch.no_grad():
        mean = kernel(Xn, gp["X"], gp["log_ls"], gp["log_sf"]) @ gp["alpha"]
    return mean.squeeze(1).numpy() * gp["y_std"] + gp["y_mean"]

In [7]:
# Out-of-fold predictions on the shared folds (hyperparameters refitted inside each fold, so the score is honest)

oof = pd.Series(np.nan, index=X.index, name="gp")
for f in range(5):
    train, val = fold != f, fold == f
    gp = fit_gp(X[train], y[train])
    oof[val] = predict_gp(gp, X[val])
    print(f"fold {f}: R2 {r2_score(y[val], oof[val]):.4f}")

print(f"out-of-fold R2: {r2_score(y, oof):.4f}")
oof.to_csv("../Stacking/oof_gp.csv")

# same folds as the SVR: compare the score, and how similar the errors are (1 = same errors, nothing new for the stack)
svr = pd.read_csv("../Stacking/oof_svr.csv", index_col="id").loc[X.index, "svr"]
print(f"SVR out-of-fold R2: {r2_score(y, svr):.4f} | correlation of GP and SVR errors: {(oof - y).corr(svr - y):.3f}")

  nll every 100 steps: [111.8, -148.4, -153.6, -154.8, -155.3, -155.5, -155.6]
fold 0: R2 0.7526
  nll every 100 steps: [102.5, -182.1, -187.4, -188.7, -189.1, -189.4, -189.5]
fold 1: R2 0.7096
  nll every 100 steps: [107.1, -173.2, -181.1, -182.7, -183.6, -183.9, -184.1]
fold 2: R2 0.6411
  nll every 100 steps: [104.4, -167.2, -174.1, -175.5, -176.2, -176.5, -176.6]
fold 3: R2 0.6622
  nll every 100 steps: [108.2, -159.2, -167.1, -169.0, -169.7, -170.0, -170.4]
fold 4: R2 0.6768
out-of-fold R2: 0.6896
SVR out-of-fold R2: 0.7083 | correlation of GP and SVR errors: 0.818


In [8]:
# Full model: refit on all 1152 rows and predict the 776 test rows (sample.csv format, for Stacking/stacking.ipynb)

gp = fit_gp(X, y)
pd.Series(predict_gp(gp, X_test), index=X_test.index, name="y").to_csv("submission_gp.csv")

# short length scale = feature the GP relies on; very long = practically ignored
length_scale = pd.Series(gp["log_ls"].exp().numpy(), index=X.columns, name="length scale").sort_values()
display(length_scale.head(10).round(2))
print(f"features with length scale > 100 (practically ignored): {(length_scale > 100).sum()} of {len(length_scale)}")

  nll every 100 steps: [105.3, -239.6, -245.8, -248.4, -250.1, -250.6, -251.0]


x641    3.26
x538    3.33
x546    3.46
x115    3.46
x362    3.63
x458    3.84
x640    4.04
x823    4.23
x531    4.73
x29     4.77
Name: length scale, dtype: float64

features with length scale > 100 (practically ignored): 124 of 216
